In [ ]:
# Cell 2 — Import + Config + Hàm xử lý DOCX

from __future__ import annotations
import os
import re
from collections import Counter
from dataclasses import dataclass
from typing import List, Iterable, Tuple

from docx import Document


@dataclass
class Config:
    # Regex drop
    drop_patterns: Tuple[re.Pattern, ...] = (
        # Page numbers
        re.compile(r"^\s*(page\s*)?\d+\s*/\s*\d+\s*$", re.I),
        re.compile(r"^\s*(page\s*)?\d+\s*$", re.I),

        # VN headings
        re.compile(r"^\s*(mục lục|lời nói đầu|lời mở đầu|tài liệu tham khảo|phụ lục)\s*$", re.I),
        re.compile(r"^\s*(chương|phần|mục)\s+([ivxlcdm]+|\d+)\b.*$", re.I),

        # Numbered headings: 1. / 1) / 1.1 / I. / A.
        re.compile(r"^\s*((\d+(\.\d+){0,6})|([ivxlcdm]+))\s*[\.\)]\s+\S.*$", re.I),


        # Captions
        # re.compile(r"^\s*(hình|figure|fig|bảng|table)\s*\d+(\.\d+)?\s*[:\-].*$", re.I),
        # re.compile(r"^\s*(nguồn|source)\s*[:\-].*$", re.I),
    )

    # Heuristic title detection
    max_title_len: int = 80
    upper_ratio_threshold: float = 0.75
    min_words_for_title: int = 2

    # Remove repeated lines (header/footer)
    repeated_min_count: int = 3
    repeated_max_len: int = 80

    # TOC detection
    toc_scan_lines: int = 200
    toc_strong_min_lines: int = 8


CFG = Config()


def normalize_spaces(s: str) -> str:
    s = (s or "").replace("\u00a0", " ")
    s = re.sub(r"[ \t]+", " ", s)
    return s.strip()


def mostly_upper(line: str, threshold: float) -> bool:
    letters = [ch for ch in line if ch.isalpha()]
    if not letters:
        return False
    upper = sum(1 for ch in letters if ch.isupper())
    return upper / len(letters) >= threshold


def looks_like_title(line: str, cfg: Config) -> bool:
    t = normalize_spaces(line)
    if not t or len(t) > cfg.max_title_len:
        return False
    words = re.findall(r"\w+", t, flags=re.UNICODE)
    if len(words) < cfg.min_words_for_title:
        return False
    return mostly_upper(t, cfg.upper_ratio_threshold)


def matches_drop(line: str, cfg: Config) -> bool:
    t = normalize_spaces(line)
    return any(pat.match(t) for pat in cfg.drop_patterns)


def toc_line_score(line: str) -> int:
    t = normalize_spaces(line)
    if not t:
        return 0
    score = 0
    if re.search(r"\.{3,}", t):          # dot leaders
        score += 2
    if re.search(r"\s\d+\s*$", t):       # ends with number
        score += 1
    if re.search(r"\b(mục|chương|phần|table of contents|contents)\b", t, re.I):
        score += 1
    return score

# --- NEW: nhận diện 1 dòng đáp án A/B/C/D ---
def is_mcq_option_line(line: str) -> bool:
    t = normalize_spaces(line)
    return bool(re.match(r"^\s*([A-Da-d])\s*([.)\:：\-])\s+.+$", t))


# --- NEW: tách A/B/C/D nếu cùng 1 dòng ---
_OPTION_MARK_RE = re.compile(r"(?i)\b([A-D])\s*([.)\:：\-])\s+")

def split_mcq_options_in_line(line: str) -> List[str]:
    t = normalize_spaces(line)
    matches = list(_OPTION_MARK_RE.finditer(t))
    if len(matches) <= 1:
        return [t] if t else [""]

    out: List[str] = []

    # Keep leading text before first option (if any)
    if matches[0].start() > 0:
        lead = t[:matches[0].start()].strip()
        if lead:
            out.append(lead)

    # Slice each option segment
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(t)
        seg = t[start:end].strip()
        if seg:
            out.append(seg)

    return out


def remove_toc_block(lines: List[str], cfg: Config) -> List[str]:
    if not lines:
        return lines

    N = min(len(lines), cfg.toc_scan_lines)
    window = lines[:N]
    strong = sum(1 for ln in window if toc_line_score(ln) >= 2)

    if strong < cfg.toc_strong_min_lines:
        return lines

    cut = 0
    for i, ln in enumerate(window):
        sc = toc_line_score(ln)
        if sc >= 2 or re.search(r"\b(mục lục|table of contents|contents)\b", ln, re.I):
            cut = i + 1
        else:
            # stop when normal text resumes
            if cut > 0 and len(normalize_spaces(ln)) > 40 and sc == 0:
                break

    return lines[cut:]


def remove_repeated_header_footer(lines: List[str], cfg: Config) -> List[str]:
    normed = [normalize_spaces(ln) for ln in lines if normalize_spaces(ln)]
    c = Counter(normed)
    repeated = {
        ln for ln, cnt in c.items()
        if cnt >= cfg.repeated_min_count and len(ln) <= cfg.repeated_max_len
    }
    return [ln for ln in lines if normalize_spaces(ln) not in repeated]


def extract_docx_paragraph_lines(docx_path: str) -> List[str]:
    doc = Document(docx_path)
    lines: List[str] = []

    for p in doc.paragraphs:
        txt = p.text or ""  # images ignored automatically
        style = (p.style.name if p.style is not None else "") or ""

        # Drop heading styles directly
        if re.search(r"\b(Heading|Title)\b", style, re.I):
            continue

        lines.append(txt)

    return lines

def clean_docx_text(docx_path: str, cfg: Config = CFG) -> str:
    lines = extract_docx_paragraph_lines(docx_path)

    lines = remove_toc_block(lines, cfg)
    lines = remove_repeated_header_footer(lines, cfg)

    out: List[str] = []
    blank = False

    for ln in lines:
        t0 = normalize_spaces(ln)

        # Tách đáp án nếu A/B/C/D cùng dòng
        pieces = split_mcq_options_in_line(t0)

        for t in pieces:
            t = normalize_spaces(t)

            if not t:
                if not blank:
                    out.append("")
                blank = True
                continue

            # ✅ Ưu tiên GIỮ đáp án A/B/C/D (không cho các rule khác xóa)
            if is_mcq_option_line(t):
                out.append(t)
                blank = False
                continue

            if matches_drop(t, cfg):
                continue

            if looks_like_title(t, cfg):
                continue

            out.append(t)
            blank = False

    text = "\n".join(out)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text



In [34]:
# Cell 3 — Test nhanh với 1 file DOCX (đổi path cho đúng)

docx_path = r"D:\data datn\test_data\Sinh_Dethi_Nguyenthihainhu_Ttgdnngdtxtpbmt.docx"  # ví dụ: r"/content/sample.docx" hoặc r"D:\data\sample.docx"

cleaned = clean_docx_text(docx_path, CFG)

print(cleaned[:2000])  # xem trước 2000 ký tự đầu
print("\n---\n")
print("Total chars:", len(cleaned))


Họ và tên giáo viên: NGUYỄN THỊ HAI NHƯ
Câu 1: Tế bào là đơn vị cấu trúc của mọi cơ thể sinh vật vì
A. mọi cơ thể sinh vật đều được cấu tạo từ một hoặc nhiều tế bào.
B. mọi cơ thể sinh vật đều được cấu tạo từ nhiều tế bào.
C. các hoạt động sống cơ bản đều được thực hiện ở tế bào, hoạt động sống ở cấp độ tế bào là nền tảng cho hoạt động sống ở cấp độ cơ thể.
D. các hoạt động sống cơ bản đều được thực hiện ở tế bào, hoạt động sống ở cấp độ cơ thể là nền tảng cho hoạt động sống ở cấp độ tế bào.
Câu 2: Bệnh nào sau đây liên quan đến sự thiếu hụt nguyên tố vi lượng?
A. Bệnh bướu cổ.
B. Bệnh còi xương.
C. Bệnh cận thị.
D. Bệnh tự kỉ.
Câu 3: Quá trình tiêu hóa thức ăn trong hệ tiêu hóa của người diễn ra theo trình tự nào dưới đây?
A. Miệng → thực quản → dạ dày → ruột non → ruột già→ hậu môn.
B. Miệng → ruột non→ dạ dày→ hầu → ruột già→ hậu môn.
C. Miệng → ruột non→ thực quản → dạ dày → ruột già → hậu môn.
D. Miệng → dạ dày → ruột non → thực quản → ruột già → hậu môn.
Câu 4: Các yếu tố chủ yếu

In [36]:
# Cell 4 — Xuất ra file .txt (để dùng tiếp)

out_txt = os.path.splitext(docx_path)[0] + ".clean.txt"
with open(out_txt, "w", encoding="utf-8") as f:
    f.write(cleaned + "\n")

out_txt


'D:\\data datn\\test_data\\Sinh_Dethi_Nguyenthihainhu_Ttgdnngdtxtpbmt.clean.txt'

In [ ]:
# Cell 5 — Batch xử lý cả folder DOCX -> output folder

def iter_docx_files(input_dir: str) -> Iterable[str]:
    for root, _, files in os.walk(input_dir):
        for fn in files:
            if fn.startswith("~$"):
                continue
            if fn.lower().endswith(".docx"):
                yield os.path.join(root, fn)

def make_out_path(input_dir: str, output_dir: str, in_path: str) -> str:
    rel = os.path.relpath(in_path, input_dir)
    base, _ = os.path.splitext(rel)
    return os.path.join(output_dir, base + ".clean.txt")

def batch_clean_docx(input_dir: str, output_dir: str, cfg: Config = CFG) -> None:
    os.makedirs(output_dir, exist_ok=True)
    total = ok = 0

    for fp in iter_docx_files(input_dir):
        total += 1
        out_fp = make_out_path(input_dir, output_dir, fp)
        os.makedirs(os.path.dirname(out_fp), exist_ok=True)

        try:
            text = clean_docx_text(fp, cfg)
            with open(out_fp, "w", encoding="utf-8") as f:
                f.write(text + "\n")
            ok += 1
        except Exception as e:
            print(f"[SKIP] {fp} -> {e}")

    print(f"Done: {ok}/{total} files cleaned. Output: {output_dir}")


# ==== chạy ====
input_dir = r"YOUR_INPUT_DIR"   # ví dụ: r"/content/docx_raw"
output_dir = r"YOUR_OUTPUT_DIR" # ví dụ: r"/content/docx_clean"

# batch_clean_docx(input_dir, output_dir, CFG)
